# Camera and depth

The robot's head camera from your own program: one picture, a stream of them, a
folder of them, then depth, which is what the camera sees in 3-D.

**None of this moves the robot, and it needs no session and no turn.** Anyone on
the Robot Network can read the camera at the same time.

Run it on the mock first: its camera draws what the head camera would see. The
mock and the simulated robot have no depth, so part 4 uses made-up points there. How to get on the Robot
Network is in the [camera quickstart](../docs/camera_quickstart.md).

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
import g1lab

# "mock" runs the robot on this laptop (needs `pip install mujoco`).
# In the lab, the robot computer: HOST = "192.168.123.222"
HOST = os.environ.get("G1LAB_HOST", "mock")
NAME = os.environ.get("G1LAB_NAME", "student")     # your name: it shows on the lab board

r = g1lab.mock(name=NAME, camera="render") if HOST == "mock" else g1lab.connect(HOST, name=NAME)
info = r.info()
print("connected to", HOST, "| session:", info.phase, "| has the robot:", info.owner)

## 1. One picture

`r.camera.read()` gives the newest picture as an array, and `r.camera.jpeg()` the
same picture as a JPEG file, ready to save.

In [ ]:
t, img = r.camera.read()
print("picture", img.shape, img.dtype)       # height, width, colours in RGB order

t, jpeg = r.camera.jpeg()
with open("view.jpg", "wb") as f:
    f.write(jpeg)
print("saved view.jpg:", len(jpeg), "bytes")

plt.figure(figsize=(6, 4.5)); plt.imshow(img); plt.title("what the robot sees"); plt.show()

On the robot the picture comes from the camera's left eye, corrected for lens
distortion. `t` is when the camera took it, by the robot's own clock, which can
be hours off from your laptop's. Use `t` to put pictures in order or measure the
time between them, not as the time of day.

## 2. Every new picture

`r.on("camera", f)` calls `f((t, picture))` on a background thread for each new
picture, and hands it the newest if your function falls behind. It returns a
function that stops it. The robot's camera sends up to 15 pictures a second.

In [ ]:
stamps = []
stop = r.on("camera", lambda m: stamps.append(m[0]))
time.sleep(3.0)
stop()

print(len(stamps), "pictures in 3 s")
print("time between pictures (s):", np.round(np.diff(stamps)[:8], 3))

## 3. A folder of pictures

For a dataset, save a picture every so often. On the robot, move the object
between pictures: different sides, distances, lighting and backgrounds make a
better dataset than many pictures of one view. To record the other sensors at the
same time, see [Recording a dataset](recording_a_dataset.ipynb).

In [ ]:
os.makedirs("pictures", exist_ok=True)
for n in range(6):                           # 6 pictures, 2 a second
    t, jpeg = r.camera.jpeg()
    with open(f"pictures/{n:03d}.jpg", "wb") as f:
        f.write(jpeg)
    time.sleep(0.5)
print(sorted(os.listdir("pictures")))

## 4. Depth

The robot's own computer works out depth from both camera eyes and sends it out
as 3-D points, 5 times a second. `r.depth.read()` gives `(t, xyz, rgb)`:

- **`xyz`** has one row per point, up to 20000, in metres from the camera's left
  eye: **x forward, y to the robot's left, z up**. The camera moves with the head,
  so these are not room coordinates.
- **`rgb`** is each point's colour (0 to 255), row for row with `xyz`.
- Only points the camera could measure are there: from about 1 ft (0.3 m) out to
  33 ft (10 m), on surfaces with some texture. Plain walls and shiny things can be
  missing.

The mock and the simulated robot have no depth, and `r.depth.read()` raises `NoData` there, so this cell
makes up a wall 1.5 m ahead with a box 1 m ahead in front of it. All the code
after this cell runs the same on real points.

In [ ]:
try:
    t, xyz, rgb = r.depth.read()
    print("real depth")
except g1lab.NoData as e:
    print(e)
    print("using made-up points instead")
    y, z = np.meshgrid(np.linspace(-1.2, 1.2, 120), np.linspace(-0.9, 0.9, 90))
    x = np.where((np.abs(y) < 0.2) & (np.abs(z) < 0.2), 1.0, 1.5)
    xyz = np.column_stack([x.ravel(), y.ravel(), z.ravel()]).astype(np.float32)
    rgb = np.full(xyz.shape, 128, np.uint8)

print(len(xyz), "points")
ahead = xyz[np.abs(xyz[:, 1]) < 0.2]         # a strip 0.4 m wide, straight ahead
if len(ahead):
    print("nearest thing ahead:", ahead[:, 0].min().round(2), "m")

## 5. Which pixel is a 3-D point

The picture and the depth points share one lens. For the 640 × 480 picture, in
pixels, the focal length is **352.1** and the centre is **325.3, 234.6**. That
puts each point on the picture, so you can find how far away something in the
picture is.

In [ ]:
F, CX, CY = 352.1, 325.3, 234.6
x, y, z = xyz[:, 0], xyz[:, 1], xyz[:, 2]
u = CX - F * y / x                           # across, from the picture's left
v = CY - F * z / x                           # down, from the picture's top

plt.figure(figsize=(6, 4.5)); plt.imshow(img)
plt.scatter(u, v, c=x, s=2, cmap="viridis"); plt.colorbar(label="distance ahead (m)")
plt.xlim(0, 640); plt.ylim(480, 0); plt.title("depth points on the picture"); plt.show()

near_centre = (np.abs(u - 320) < 20) & (np.abs(v - 240) < 20)     # a box around a pixel
if near_centre.any():
    print("the middle of the picture is", np.median(x[near_centre]).round(2), "m ahead")

Going the other way, pixel `(u, v)` looks along the direction
`(1, (CX - u) / F, (CY - v) / F)`: forward, left, up.

**Try this.** In the lab, find a coloured object in the picture with `find_ball()`
from [Perception](perception.ipynb) and use the box above to ask depth how far away
it is.

In [ ]:
r.close()